# 13 — Experiment 1: cross-sectional donors with semantic parcel alignment

Donors = the collaborator's 5 covariate-matched controls per treated site; cloud handling =
chip-mean fill cache (as every previous experiment); outcome = the 980-d latent
(5 ch × 196 parcels), **donor parcels permuted into the treated layout** by the notebook-12
alignment. Scorer = notebook 11 (simplex SCM via SLSQP, missing rows dropped, per-site
standardized test RMSE with train alongside, plain mean over 10 sites, S1 before S2; P09 with
P01–P08, P10 with P01–P09).

Standardization: the notebook-11 scaler is per (channel, position). A permuted donor moves
parcels between positions, so aligned arms use a **per-channel** scaler (pooled over all sites,
training periods and parcels; 5 means / 5 SDs). Arm 1-A is reported under both scalers — the
per-dimension one reproduces the notebook-11 `latent980` rows.

| arm | unit | alignment |
|---|---|---|
| 1-A | site | none (same coordinate) |
| 1-B | site | class only |
| 1-C | site | class + elevation/slope |
| 1-D | site | + TerraMind history (minimal D=7 / full D=29 descriptor) |
| 1-E | site | full descriptor + 3×3 context |
| 1-F | site | pooled reps (chip-mean / quantile / Gram) of the aligned donors — sanity |
| 1-G | site | 2×2 block means of aligned parcels (245-d) — per-parcel-noise control |
| 1-P | parcel | per treated position: K = 10 nearest same-class donor parcels, simplex (and ridge λ = 1) |

C1 = test ≤ 1.5 × train (collaborator's flag); C2 = test error < own P01–P08 average; C3 = test
error < equal-weight donor average — all in the same standardized 980-d space.


In [1]:
import sys, itertools
import numpy as np, pandas as pd
sys.path.insert(0, ".")
import panel_lib as pl, panel_repr as pr, panel_align as pa
pd.set_option("display.width", 220)

panel = pl.Panel.from_npz(pl.LATD / "latents_biweekly.npz")
VAL = pr.load_validity()
ALL_SITES = sorted(panel.roster["site_id"]); TREAT = panel.treatments; DON = pr.matched_donors(panel)
DESC = pa.load_descriptors("parcel_descriptors.npz"); S = pa.standardize(DESC)
_z = np.load("parcel_perms.npz"); PERMS = {tuple(k.split("|")): _z[k] for k in _z.files}
IDENT = np.arange(196)
TESTS = ((9, range(1, 9)), (10, range(1, 10)))


def scaler(sensor, train_p, mode):
    T = np.stack([panel.L(s, sensor, q) for s in ALL_SITES for q in train_p if panel.L(s, sensor, q) is not None])
    if mode == "perdim":                                  # notebook 11
        mu, sd = T.mean(0), T.std(0, ddof=1)
    else:                                                 # per channel, pooled over parcels
        V = T.reshape(len(T), 5, 196).transpose(1, 0, 2).reshape(5, -1)
        mu, sd = np.repeat(V.mean(1), 196), np.repeat(V.std(1, ddof=1), 196)
    sd[~np.isfinite(sd) | (sd == 0)] = 1.0
    return lambda v: (v - mu) / sd


def vec(site, sensor, q, perm=None):
    if perm is None:
        v = panel.L(site, sensor, q)
        return np.full(980, np.nan) if v is None else v
    return pa.aligned_vec(panel, site, sensor, q, perm)


def validate_site(sensor, t, perm_of, z, transform, train_p, test_q, ridge=0.0):
    """perm_of(j) -> permutation (or None to drop donor j). transform: 980 -> M dims."""
    dl = [j for j in DON[t] if perm_of(j) is not None]
    f = lambda s, q, p: transform(z(vec(s, q=q, sensor=sensor, perm=p)))
    ytr = np.concatenate([f(t, q, None) for q in train_p])
    Xtr = np.column_stack([np.concatenate([f(j, q, perm_of(j)) for q in train_p]) for j in dl])
    w = pa.simplex_scm(ytr, Xtr, ridge)
    yte = f(t, test_q, None); Xte = np.column_stack([f(j, test_q, perm_of(j)) for j in dl])
    own = np.nanmean(np.stack([f(t, q, None) for q in train_p]), 0)
    tr, te = pa.rmse(ytr - Xtr @ w), pa.rmse(yte - Xte @ w)
    c2, c3 = pa.rmse(yte - own), pa.rmse(yte - np.nanmean(Xte, 1))
    return {"site": t, "test": f"P{test_q:02d}", "test_rmse": te, "train_rmse": tr, "own_hist_rmse": c2,
            "equal_w_rmse": c3, "C1": te <= 1.5 * tr, "C2": te < c2, "C3": te < c3,
            "n_donors": len(dl), "n_test_dims": int(np.isfinite(yte - Xte @ w).sum()), "w_max": float(w.max())}


def run_arm(label, sensor, perm_fn, transform=lambda v: v, mode="perchannel", ridge=0.0):
    rows = []
    for test_q, train_p in TESTS:
        z = scaler(sensor, train_p, mode)
        for t in TREAT:
            r = validate_site(sensor, t, lambda j: perm_fn(sensor, t, j), z, transform, train_p, test_q, ridge)
            r.update(arm=label, sensor=sensor); rows.append(r)
    return pd.DataFrame(rows)


def summarize(df):
    g = df.groupby(["sensor", "arm", "test"])
    out = g[["test_rmse", "train_rmse"]].mean().round(3)
    out["C1"] = g.C1.sum(min_count=1); out["C2"] = g.C2.sum(min_count=1); out["C3"] = g.C3.sum(min_count=1); out["n"] = g.size()
    return out


def perm_arm(arm):
    def f(sensor, t, j):
        if arm == "A": return IDENT
        p = PERMS[(arm, sensor, t, j)]
        return p if (p >= 0).sum() >= pa.MIN_MATCHED else None
    return f
print("ready")


ready


## 1. Site-level arms A–E (both sensors, P09 and P10)

mean standardized test RMSE (train), pass counts over 10 sites.

In [2]:
SITE = []
SITE.append(run_arm("A_perdim(nb11)", "sentinel1", perm_arm("A"), mode="perdim"))
SITE.append(run_arm("A_perdim(nb11)", "sentinel2", perm_arm("A"), mode="perdim"))
for sensor in pl.SENSORS:
    for arm in ["A", "B", "C", "D_min", "D_full", "E"]:
        SITE.append(run_arm(arm, sensor, perm_arm(arm)))
SITE = pd.concat(SITE, ignore_index=True)
ORDER = ["A_perdim(nb11)", "A", "B", "C", "D_min", "D_full", "E"]
for sensor in pl.SENSORS:
    print(f"\n=== {sensor} ===")
    print(summarize(SITE.query("sensor == @sensor")).reset_index().set_index("arm").loc[ORDER]
          .set_index("test", append=True).to_string())
ref = pd.read_csv("panel_collabstyle_scm_validation.csv").query("cache == 'chipmean' and repr == 'latent980'")
print("\nnotebook-11 latent980 rows (must equal A_perdim):")
print(ref[["sensor", "test", "mean_test_rmse", "mean_train_rmse"]].round(3).to_string(index=False))


/tmp/ipykernel_1805750/1504401592.py:44: RuntimeWarning: Mean of empty slice
  c2, c3 = pa.rmse(yte - own), pa.rmse(yte - np.nanmean(Xte, 1))



=== sentinel1 ===
                        sensor  test_rmse  train_rmse  C1  C2  C3   n
arm            test                                                  
A_perdim(nb11) P09   sentinel1      1.084       1.093  10   0  10  10
               P10   sentinel1      1.087       1.093  10   0   8  10
A              P09   sentinel1      1.079       1.088  10   0  10  10
               P10   sentinel1      1.082       1.088  10   0   7  10
B              P09   sentinel1      1.088       1.087  10   0  10  10
               P10   sentinel1      1.089       1.087  10   0   8  10
C              P09   sentinel1      1.086       1.074  10   0   9  10
               P10   sentinel1      1.077       1.076  10   0  10  10
D_min          P09   sentinel1      0.791       0.734  10   0   9  10
               P10   sentinel1      0.797       0.742  10   0  10  10
D_full         P09   sentinel1      0.805       0.763  10   0  10  10
               P10   sentinel1      0.831       0.769  10   0   9  10
E

### Per-site view, arm D (minimal) vs A — where does alignment move the error?

In [3]:
for sensor in pl.SENSORS:
    a = SITE.query("sensor == @sensor and arm in ['A', 'D_min'] and test == 'P09'").pivot_table(index="site", columns="arm", values="test_rmse")
    a["delta"] = a["D_min"] - a["A"]
    a["same_pos_agree(mean of 5 donors)"] = [pd.read_csv("panel_align_agreement.csv").query("treated == @s").same_position_agreement.mean().round(2) for s in a.index]
    print(f"\n{sensor}, P09:"); print(a.round(3).to_string())



sentinel1, P09:
arm                 A  D_min  delta  same_pos_agree(mean of 5 donors)
site                                                                 
treatment_0001  1.104  0.900 -0.204                              0.37
treatment_0002  1.068  0.801 -0.268                              0.29
treatment_0003  1.072  0.824 -0.248                              0.53
treatment_0004  1.081  0.681 -0.400                              0.21
treatment_0005  1.047  0.810 -0.237                              0.34
treatment_0006  1.095  0.882 -0.213                              0.63
treatment_0007  1.086  0.729 -0.357                              0.75
treatment_0008  1.055  0.827 -0.228                              0.51
treatment_0009  1.091  0.736 -0.355                              0.63
treatment_0010  1.089  0.721 -0.368                              0.43

sentinel2, P09:
arm                 A  D_min  delta  same_pos_agree(mean of 5 donors)
site                                                    

## 2. Sanity 1-F (pooled reps are permutation-invariant) and control 1-G (2×2 block means)

In [4]:
def pooled(name):
    def f(v):
        A = v.reshape(5, 196); ok = np.isfinite(A).all(0); A = A[:, ok]
        if A.shape[1] == 0: return np.full(pr.dim_of(name), np.nan)
        return pr._repr_raw(A, name)
    return f
# 1-F: pooled reps computed on the RAW (unstandardized) aligned parcels, then z-scored per repr dim as in nb 11
def run_pooled(label, sensor, perm_fn, name):
    rows = []
    for test_q, train_p in TESTS:
        F = {}
        for s in ALL_SITES:
            for q in range(1, 11):
                F[(s, q)] = pooled(name)(vec(s, sensor, q))
        T = np.stack([F[(s, q)] for s in ALL_SITES for q in train_p])
        mu, sd = np.nanmean(T, 0), np.nanstd(T, 0, ddof=1); sd[~np.isfinite(sd) | (sd == 0)] = 1.0
        z = lambda v: (v - mu) / sd
        for t in TREAT:
            dl = DON[t]
            g = lambda s, q, p: z(pooled(name)(vec(s, sensor, q, p)))
            ytr = np.concatenate([g(t, q, None) for q in train_p])
            Xtr = np.column_stack([np.concatenate([g(j, q, perm_fn(sensor, t, j)) for q in train_p]) for j in dl])
            w = pa.simplex_scm(ytr, Xtr)
            yte = g(t, test_q, None); Xte = np.column_stack([g(j, test_q, perm_fn(sensor, t, j)) for j in dl])
            rows.append({"arm": label, "sensor": sensor, "site": t, "test": f"P{test_q:02d}",
                         "test_rmse": pa.rmse(yte - Xte @ w), "train_rmse": pa.rmse(ytr - Xtr @ w)})
    return pd.DataFrame(rows)

F_ROWS = []
for sensor in pl.SENSORS:
    for name in ["chip_mean", "gram"]:
        F_ROWS.append(run_pooled(f"F_{name}_ident", sensor, lambda se, t, j: IDENT, name))
        F_ROWS.append(run_pooled(f"F_{name}_D_min", sensor, lambda se, t, j: PERMS[("D_min", se, t, j)], name))
FP = pd.concat(F_ROWS, ignore_index=True)
print("1-F: pooled reps, identity perm (must equal notebook 11) vs arm-D perm (differs only through unmatched parcels dropped):")
print(FP.groupby(["sensor", "arm", "test"])[["test_rmse", "train_rmse"]].mean().round(3).to_string())
ref = pd.read_csv("panel_collabstyle_scm_validation.csv").query("cache == 'chipmean' and repr in ['chip_mean', 'gram']")
print("\nnotebook-11 reference:"); print(ref[["sensor", "repr", "test", "mean_test_rmse", "mean_train_rmse"]].round(3).to_string(index=False))

G = []
for sensor in pl.SENSORS:
    for arm in ["A", "D_min"]:
        G.append(run_arm(f"G_block2_{arm}", sensor, perm_arm(arm), transform=pa.block_mean))
G = pd.concat(G, ignore_index=True)
print("\n1-G: 2x2 block means (245-d), per-channel scaler:"); print(summarize(G).to_string())
SITE = pd.concat([SITE, G], ignore_index=True)


1-F: pooled reps, identity perm (must equal notebook 11) vs arm-D perm (differs only through unmatched parcels dropped):
                                  test_rmse  train_rmse
sensor    arm               test                       
sentinel1 F_chip_mean_D_min P09       0.627       0.587
                            P10       0.682       0.595
          F_chip_mean_ident P09       0.603       0.587
                            P10       0.701       0.596
          F_gram_D_min      P09       0.740       0.668
                            P10       0.733       0.676
          F_gram_ident      P09       0.692       0.659
                            P10       0.732       0.665
sentinel2 F_chip_mean_D_min P09       0.622       0.745
                            P10       0.398       0.744
          F_chip_mean_ident P09       0.577       0.741
                            P10       0.395       0.736
          F_gram_D_min      P09       0.683       0.746
                            P10       0

/tmp/ipykernel_1805750/1504401592.py:44: RuntimeWarning: Mean of empty slice
  c2, c3 = pa.rmse(yte - own), pa.rmse(yte - np.nanmean(Xte, 1))



1-G: 2x2 block means (245-d), per-channel scaler:
                               test_rmse  train_rmse  C1  C2  C3   n
sensor    arm            test                                       
sentinel1 G_block2_A     P09       0.567       0.578  10   0   8  10
                         P10       0.569       0.577  10   0  10  10
          G_block2_D_min P09       0.428       0.415  10   0  10  10
                         P10       0.440       0.417  10   0  10  10
sentinel2 G_block2_A     P09       0.784       0.700  10   1   2  10
                         P10       0.804       0.702  10   2   5  10
          G_block2_D_min P09       0.726       0.642  10   6   8  10
                         P10       0.589       0.645  10   9   7  10


## 3. Parcel-level arm 1-P — each treated position gets its own weights over the K = 10 nearest same-class donor parcels

Distance = arm-D (minimal) cost. Rows per fit = training periods × 5 channels. Positions with fewer than 3 candidates are left unpredicted (NaN).

In [5]:
K, MINPOOL = 10, 3
def run_parcel(sensor, ridge=0.0, K=K):
    rows = []
    for test_q, train_p in TESTS:
        z = scaler(sensor, train_p, "perchannel")
        Zt = {s: {q: (z(vec(s, sensor, q)).reshape(5, 196)) for q in range(1, 11)} for s in ALL_SITES}
        for t in TREAT:
            dt = S[(t, sensor)]
            cands = []   # (cost row block, donor id)
            for j in DON[t]:
                C = pa.cost_matrix(dt, S[(j, sensor)], alpha=1.0, beta=1.0, variant="minimal")
                cands.append((C, j))
            Call = np.concatenate([c for c, _ in cands], axis=1)          # (196, 5*196)
            who = [(j, q) for _, j in cands for q in range(196)]
            pred_te = np.full((5, 196), np.nan); pred_tr = []; y_tr_all = []
            for p in range(196):
                order = np.argsort(Call[p]); order = order[Call[p, order] < pa.BIG][:K]
                if len(order) < MINPOOL: continue
                y = np.concatenate([Zt[t][q][:, p] for q in train_p])
                X = np.column_stack([np.concatenate([Zt[who[k][0]][q][:, who[k][1]] for q in train_p]) for k in order])
                if not np.isfinite(y).any(): continue
                w = pa.simplex_scm(y, X, ridge)
                pred_tr.append(X @ w); y_tr_all.append(y)
                pred_te[:, p] = np.column_stack([Zt[who[k][0]][test_q][:, who[k][1]] for k in order]) @ w
            yte = Zt[t][test_q]
            e_tr = np.concatenate(y_tr_all) - np.concatenate(pred_tr)
            te, tr = pa.rmse((yte - pred_te).ravel()), pa.rmse(e_tr)
            own = np.nanmean(np.stack([Zt[t][q] for q in train_p]), 0)
            rows.append({"arm": f"P_K{K}" + ("_ridge1" if ridge else ""), "sensor": sensor, "site": t, "test": f"P{test_q:02d}",
                         "test_rmse": te, "train_rmse": tr, "own_hist_rmse": pa.rmse((yte - own).ravel()),
                         "C1": te <= 1.5 * tr, "C2": te < pa.rmse((yte - own).ravel()), "C3": np.nan,
                         "n_pred_positions": int(np.isfinite(pred_te[0]).sum())})
    return pd.DataFrame(rows)

P = pd.concat([run_parcel(s, r) for s in pl.SENSORS for r in (0.0, 1.0)], ignore_index=True)
g = P.groupby(["sensor", "arm", "test"])
out = g[["test_rmse", "train_rmse"]].mean().round(3); out["C1"] = g.C1.sum(); out["C2"] = g.C2.sum(); out["pred_pos"] = g.n_pred_positions.mean().round(0)
print("1-P parcel-level (per-channel scaler):"); print(out.to_string())
SITE = pd.concat([SITE, P], ignore_index=True)


1-P parcel-level (per-channel scaler):
                             test_rmse  train_rmse  C1  C2  pred_pos
sensor    arm          test                                         
sentinel1 P_K10        P09       0.785       0.630   9   0     195.0
                       P10       0.796       0.642  10   0     195.0
          P_K10_ridge1 P09       0.763       0.657  10   0     195.0
                       P10       0.772       0.667  10   0     195.0
sentinel2 P_K10        P09       1.053       0.739   4   2     180.0
                       P10       0.899       0.755   9   8     180.0
          P_K10_ridge1 P09       1.007       0.769   9   3     180.0
                       P10       0.869       0.784  10   9     180.0


## 4. Sensitivity of arm D (minimal) to α (physical) and β (TerraMind history)

In [6]:
SENS = []
for sensor in pl.SENSORS:
    for a, b in itertools.product([0.3, 1.0, 3.0], [0.3, 1.0, 3.0]):
        pf = lambda se, t, j, a=a, b=b: pa.align_pair(S[(t, se)], S[(j, se)], alpha=a, beta=b, variant="minimal")
        df = run_arm(f"D_min_a{a}_b{b}", sensor, pf); df["alpha"] = a; df["beta"] = b; SENS.append(df)
SENS = pd.concat(SENS, ignore_index=True)
print(SENS.groupby(["sensor", "test", "alpha", "beta"]).test_rmse.mean().round(3).unstack("beta").to_string())


/tmp/ipykernel_1805750/1504401592.py:44: RuntimeWarning: Mean of empty slice
  c2, c3 = pa.rmse(yte - own), pa.rmse(yte - np.nanmean(Xte, 1))


/tmp/ipykernel_1805750/1504401592.py:44: RuntimeWarning: Mean of empty slice
  c2, c3 = pa.rmse(yte - own), pa.rmse(yte - np.nanmean(Xte, 1))


/tmp/ipykernel_1805750/1504401592.py:44: RuntimeWarning: Mean of empty slice
  c2, c3 = pa.rmse(yte - own), pa.rmse(yte - np.nanmean(Xte, 1))


/tmp/ipykernel_1805750/1504401592.py:44: RuntimeWarning: Mean of empty slice
  c2, c3 = pa.rmse(yte - own), pa.rmse(yte - np.nanmean(Xte, 1))


/tmp/ipykernel_1805750/1504401592.py:44: RuntimeWarning: Mean of empty slice
  c2, c3 = pa.rmse(yte - own), pa.rmse(yte - np.nanmean(Xte, 1))


...

/tmp/ipykernel_1805750/1504401592.py:44: RuntimeWarning: Mean of empty slice
  c2, c3 = pa.rmse(yte - own), pa.rmse(yte - np.nanmean(Xte, 1))


/tmp/ipykernel_1805750/1504401592.py:44: RuntimeWarning: Mean of empty slice
  c2, c3 = pa.rmse(yte - own), pa.rmse(yte - np.nanmean(Xte, 1))


/tmp/ipykernel_1805750/1504401592.py:44: RuntimeWarning: Mean of empty slice
  c2, c3 = pa.rmse(yte - own), pa.rmse(yte - np.nanmean(Xte, 1))


/tmp/ipykernel_1805750/1504401592.py:44: RuntimeWarning: Mean of empty slice
  c2, c3 = pa.rmse(yte - own), pa.rmse(yte - np.nanmean(Xte, 1))


beta                    0.3    1.0    3.0
sensor    test alpha                     
sentinel1 P09  0.3    0.802  0.787  0.783
               1.0    0.776  0.802  0.784
               3.0    0.802  0.778  0.802
          P10  0.3    0.792  0.788  0.777
               1.0    0.798  0.792  0.786
               3.0    0.833  0.792  0.792
sentinel2 P09  0.3    0.980  0.978  0.959
               1.0    0.965  0.980  0.966
               3.0    0.975  0.959  0.980
          P10  0.3    0.891  0.884  0.872
               1.0    0.906  0.891  0.885
               3.0    0.939  0.912  0.891


In [7]:
SITE.to_csv("panel_align_exp1_sites.csv", index=False)
MEAN = summarize(SITE).reset_index(); MEAN.to_csv("panel_align_exp1_validation.csv", index=False)
print("saved panel_align_exp1_validation.csv / panel_align_exp1_sites.csv")
print(MEAN.pivot_table(index=["sensor", "arm"], columns="test", values=["test_rmse", "train_rmse", "C2", "C3"]).round(3).to_string())


saved panel_align_exp1_validation.csv / panel_align_exp1_sites.csv
                           C2         C3       test_rmse        train_rmse       
test                      P09  P10   P09   P10       P09    P10        P09    P10
sensor    arm                                                                    
sentinel1 A               0.0  0.0  10.0   7.0     1.079  1.082      1.088  1.088
          A_perdim(nb11)  0.0  0.0  10.0   8.0     1.084  1.087      1.093  1.093
          B               0.0  0.0  10.0   8.0     1.088  1.089      1.087  1.087
          C               0.0  0.0   9.0  10.0     1.086  1.077      1.074  1.076
          D_full          0.0  0.0  10.0   9.0     0.805  0.831      0.763  0.769
          D_min           0.0  0.0   9.0  10.0     0.791  0.797      0.734  0.742
          E               0.0  0.0  10.0   9.0     0.809  0.841      0.767  0.774
          G_block2_A      0.0  0.0   8.0  10.0     0.567  0.569      0.578  0.577
          G_block2_D_min  0.0  

## Reading

1. **Class alone does nothing; class + TerraMind history does.** Arms B and C stay at the
   same-coordinate level (Sentinel-1 ≈ 1.08, Sentinel-2 ≈ 1.05–1.15). Adding the parcel's
   pre-treatment latent history to the cost (arm D, minimal descriptor) lowers the 980-d error to
   **Sentinel-1 0.791 (0.734) / 0.797 (0.742)** and **Sentinel-2 0.992 (0.813) / 0.903 (0.826)**
   at P09 / P10 — a 27 % / 10–27 % reduction over arm A. The full descriptor and the context term
   (D_full, E) are slightly worse than minimal; the α/β grid moves Sentinel-1 by ≤ 0.03 and
   Sentinel-2 by ≤ 0.04, so the result is not tuned.
2. **Gains are not confined to mixed sites** — the stated prediction was wrong. Sentinel-1
   improves at every site by 0.20–0.40, including the forest-dominated 0007 (agreement 0.75).
   Within a class, choosing the donor parcel whose history resembles the treated parcel's is what
   helps; correcting the class map by itself is not enough (arm B).
3. **Aligned 980-d still loses to pooled representations, except Sentinel-1 block means.**
   Chip-mean pooling gives Sentinel-1 0.603 / 0.701 and Sentinel-2 0.577 / 0.395 (notebook 11).
   Arm G (2×2 block means of the aligned parcels, 245-d) reaches **Sentinel-1 0.428 / 0.440** —
   the best Sentinel-1 latent number so far, and C3 10/10 — while on Sentinel-2 (0.726 / 0.589)
   it stays behind chip-mean pooling. Per-parcel FSQ noise, not correspondence, is what remains
   in the 980-d arms: averaging 4 aligned parcels removes more error than any cost term.
4. **Parcel-level weights (1-P) do not beat one global weight per donor.** K = 10 same-class
   nearest parcels, simplex: Sentinel-1 0.785 / 0.796, Sentinel-2 1.053 / 0.899; ridge λ = 1
   0.763 / 0.772 and 1.007 / 0.869. Sentinel-2 P09 fails C1 at 6 of 10 sites (train 0.74 vs test
   1.05) — position-wise fits overfit the 40 training rows.
5. **Donor sufficiency (C2/C3).** Sentinel-1: no arm beats the site's own P01–P08 average (C2
   0/10 everywhere) although all beat the equal-weight donor average. Sentinel-2: arm D_min passes
   C2 at 5/10 (P09) and 8/10 (P10) sites, up from 1/10 and 0/10 for arm A.
6. Sanity gates passed: arm A with the notebook-11 scaler reproduces the `latent980` rows
   (1.084/1.087, 1.107/1.251); identity-permuted pooled reps reproduce notebook 11 to 3 decimals.
   Aligned pooled reps differ only through unmatched parcels being dropped (+0.02–0.06).
